In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

seoul_bus_key = os.environ.get("SEOUL_BUS_API_KEY")
seoul_subway_key = os.environ.get("SEOUL_SUBWAY_API_KEY")
weather_key = os.environ.get("WEATHER_API_KEY")

In [4]:
from datetime import datetime, timedelta
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()

current_path =os.getcwd()
root = os.path.dirname(os.path.dirname(current_path))
# raw_folder = os.path.join("hdfs:///user/maria_dev/BDP/data/raw")
seoul_bus_key = os.environ.get("SEOUL_BUS_API_KEY")
current_path =os.getcwd()
raw_folder = os.path.join(os.getcwd(),"data","raw")
# file_path = os.path.join(raw_folder,f"weather_data_{year}.csv")

def get_bus_data(target_day):

    start=1
    end = 1000
    all_data = []
    while True:
        url = base_url = f"http://openapi.seoul.go.kr:8088/{seoul_bus_key}/json/CardBusStatisticsServiceNew/{start}/{end}/{target_day}"
        response = requests.get(url)
        data = response.json()
        if start ==1:
            total_count=data["CardBusStatisticsServiceNew"]['list_total_count']
            print(f"총 데이터 수: {total_count}")
        
        rows = data["CardBusStatisticsServiceNew"]['row']
        all_data.extend(rows)
        
        if len(rows)<1000 or (start +len(rows)-1)>=total_count:
            break

        start += 1000
        end += 1000

        # time.sleep(0.5)
    
    if all_data:
        df = pd.DataFrame(all_data)
        return df
    
def get_bus_data_range(start_date, end_date, weekend_only = False):
    start = datetime.strptime(start_date, '%Y%m%d')
    end = datetime.strptime(end_date, '%Y%m%d')
    
    date_list=[]
    df_list =[]

    for i in range((end-start).days +1):
        date_list.append((start+timedelta(days=i)).strftime("%Y%m%d"))

    for date in date_list:
        if weekend_only:
            date_obj = datetime.strptime(date, "%Y%m%d")
            #weekday()의 결과는 0(월)부터 6(일)
            if date_obj.weekday()<5:
                continue
        df = get_bus_data(date)

        if df is not None:
            df_list.append(df)
            print(f"{date} 수집완료")
        else:
            print(f"{date} 데이터 형식 오류, {df}")
    if(df_list):
        result_df = pd.concat(df_list, ignore_index=True)
        return result_df
    else:
        return None

df = get_bus_data_range("20250101","20250102", weekend_only=False)
file_path = os.path.join(raw_folder,f"BUS_STATION_BOARDING_MONTH_202412.csv")

df.to_csv(f"{file_path}",index=False, encoding='utf-8')

print("저장완료")



총 데이터 수: 40012
20250101 수집완료
총 데이터 수: 41145
20250102 수집완료


OSError: Cannot save file into a non-existent directory: 'c:\Users\jysh0\Desktop\BDP\src\ingest\data\raw'

In [ ]:

load_dotenv()
def get_subway_data(start_date, end_date):
    
    
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    date_list =[]
    for i in range((end-start).days+1):
        date_list.append((start+timedelta(days=i)).strftime("%Y%m%d"))
    
    all_data = []
    for date in date_list:
        url = f"http://openapi.seoul.go.kr:8088/{seoul_subway_key}/json/CardSubwayStatsNew/1/1000/{date}"#+{START_DATE}+{END_DATE}"
        response = requests.get(url)
        data = response.json()

        if "RESULT" in data:
            message = data["RESULT"].get("MESSAGE")
            code = data["RESULT"].get("CODE")
            print(f"[{date}] 서버 응답: {code} - {message}")
            print(url)

        if "CardSubwayStatsNew" in data and "row" in data["CardSubwayStatsNew"]:

            rows = data["CardSubwayStatsNew"]["row"]
            all_data.extend(rows)
        

    df = pd.DataFrame(all_data)
    return df

print(get_subway_data("20230101", "20230131"))



[20230101] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088/4c6556784d6a79733737564e684866/json/CardSubwayStatsNew/1/5/20151101/9호선/김포공항
[20230102] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088/4c6556784d6a79733737564e684866/json/CardSubwayStatsNew/1/5/20151101/9호선/김포공항
[20230103] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088/4c6556784d6a79733737564e684866/json/CardSubwayStatsNew/1/5/20151101/9호선/김포공항
[20230104] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088/4c6556784d6a79733737564e684866/json/CardSubwayStatsNew/1/5/20151101/9호선/김포공항
[20230105] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088/4c6556784d6a79733737564e684866/json/CardSubwayStatsNew/1/5/20151101/9호선/김포공항
[20230106] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088/4c6556784d6a79733737564e684866/json/CardSubwayStatsNew/1/5/20151101/9호선/김포공항
[20230107] 서버 응답: INFO-200 - 해당하는 데이터가 없습니다.
http://openapi.seoul.go.kr:8088

In [104]:
import io

cols = ["TM","STN","WS_AVG","WR_DAY","WD_MAX","WS_MAX","WS_MAX_TM","WD_INS",
        "WS_INS","WS_INS_TM","TA_AVG","TA_MAX","TA_MAX_TM","TA_MIN","TA_MIN_TM",
        "TD_AVG","TS_AVG","TG_MIN","HM_AVG","HM_MIN","HM_MIN_TM","PV_AVG","EV_S",
        "EV_L","FG_DUR","PA_AVG","PS_AVG","PS_MAX","PS_MAX_TM","PS_MIN","PS_MIN_TM",
        "CA_TOT","SS_DAY","SS_DUR","SS_CMB","SI_DAY","SI_60M_MAX","SI_60M_MAX_TM",
        "RN_DAY","RN_D99","RN_DUR","RN_60M_MAX","RN_60M_MAX_TM","RN_10M_MAX","RN_10M_MAX_TM",
        "RN_POW_MAX","RN_POW_MAX_TM","SD_NEW","SD_NEW_TM","SD_MAX","SD_MAX_TM",
        "TE_05","TE_10","TE_15","TE_30","TE_50"
]
#일자료
def get_weather_data(target_date):
    
    try:
        url = f"https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd.php?tm={target_date}&stn=0&help=1&authKey={weather_key}"
        response = requests.get(url)
        content = response.text.strip()

        if not content or all(line.startswith('#') for line in content.splitlines()):
            return None

    except Exception as e:
        print(f"날씨 데이터 요청 중 오류 발생: {e}")
        
    
    df = pd.read_csv(io.StringIO(response.text), comment='#', header=None)
    if df.iloc[:,-1].isnull().all():
        df = df.iloc[:,:-1] #마지막열 삭제(결측치)
    df.columns = cols
    # df =response.text

    return df
def get_weather_data_range(start_date, end_date):
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    date_list =[]
    df_list=[]
    for i in range((end-start).days+1):
        date_list.append((start+timedelta(days=i)).strftime("%Y%m%d"))

    for date in date_list:
        df =get_weather_data(date)
        if df is not None and len(df.columns) == len(cols):
            df_list.append(df)
            print(f"{date} 수집완료")
        else:
            print(f"{date} 데이터 형식 오류, {df}")
    if(df_list):
        result_df = pd.concat(df_list, ignore_index=True)
        return result_df

result = get_weather_data_range("20240101", "20240131")
print(result.head())
result.to_csv("weather_data_2024_01.csv", index=False, encoding='utf-8')


20240101 수집완료
20240102 수집완료
20240103 수집완료
20240104 수집완료
20240105 수집완료
20240106 수집완료
20240107 수집완료
20240108 수집완료
20240109 수집완료
20240110 수집완료
20240111 수집완료
20240112 수집완료
20240113 수집완료
20240114 수집완료
20240115 수집완료
20240116 수집완료
20240117 수집완료
20240118 수집완료
20240119 수집완료
20240120 수집완료
20240121 수집완료
20240122 수집완료
20240123 수집완료
20240124 수집완료
20240125 수집완료
20240126 수집완료
20240127 수집완료
20240128 수집완료
20240129 수집완료
20240130 수집완료
20240131 수집완료
         TM  STN  WS_AVG  WR_DAY  WD_MAX  WS_MAX  WS_MAX_TM  WD_INS  WS_INS  \
0  20240101   90     1.0     900      14     2.3       1440      29     4.5   
1  20240101   93     0.6     501      23     1.6        308      18     2.4   
2  20240101   95     0.7     570      20     2.4       1616      23     4.7   
3  20240101   98     0.8     673      18     2.4       1438      16     4.4   
4  20240101   99     0.6     540      27     1.7       1525      27     3.1   

   WS_INS_TM  ...  RN_POW_MAX_TM  SD_NEW  SD_NEW_TM  SD_MAX  SD_MAX_TM  TE_05  \
0       22

In [ ]:
import subprocess

def load_to_hdfs(local_path, hdfs_path):
    cmd = f"hdfs fs -put {local_path} {hdfs_path}"
    try:
        subprocess.run(cmd, shell=True, check = True)
        print(f"{local_path} 업로드 완료")
    except subprocess.CalledProcessError as e:
        print(f"HDFS 업로드 실패: {e}")

local_path = "weather_data_2024_01.csv"
hdfs_path = ""